In [1]:
# ── Cell 1 of Aggregate.ipynb ────────────────────────────────
#
# PORTABLE sys.path BOOTSTRAP
# This notebook lives at aggregation/Aggregate.ipynb
# → project root is one level up from os.getcwd()
#
import sys, os

_HERE         = os.path.abspath(os.getcwd())   # .../project/aggregation
_PROJECT_ROOT = os.path.dirname(_HERE)          # .../project
if _PROJECT_ROOT not in sys.path:
    sys.path.insert(0, _PROJECT_ROOT)

import pickle, shutil
import tenseal as ts

from config import (
    RESULTS_PATH,
    CHUNKS_FOLDER,        # server/received_chunks_bin
    AGG_CHUNKS_FOLDER,    # server/aggregated_chunks_bin
    AGGREGATED_GRAD_FILE, # project_root/aggregated_gradient_global_encrypted.pkl
)


def aggregate_encrypted_gradients():
    # ── Load CKKS context ─────────────────────────────────────
    context_path = os.path.join(RESULTS_PATH, "context.ser")
    with open(context_path, "rb") as f:
        context = ts.context_from(f.read())
    print(f"[AGG] CKKS context loaded from {context_path}")

    # ── Discover client chunk directories ─────────────────────
    if not os.path.exists(CHUNKS_FOLDER):
        print(f"[AGG] No received_chunks_bin folder found at {CHUNKS_FOLDER}")
        return

    client_dirs = [
        d for d in os.listdir(CHUNKS_FOLDER)
        if os.path.isdir(os.path.join(CHUNKS_FOLDER, d))
    ]
    if not client_dirs:
        print("[AGG] No client gradients to aggregate.")
        return
    print(f"[AGG] Found {len(client_dirs)} client(s): {client_dirs}")

    # ── Load encrypted chunks per client ──────────────────────
    chunk_groups: dict[int, list] = {}

    for client_id in client_dirs:
        client_path = os.path.join(CHUNKS_FOLDER, client_id)
        chunk_files = sorted(
            [f for f in os.listdir(client_path) if f.endswith(".bin")],
            key=lambda x: int(x.split("_")[-1].split(".")[0]),
        )
        for fname in chunk_files:
            try:
                idx = int(fname.split("_")[-1].split(".")[0])
                with open(os.path.join(client_path, fname), "rb") as f:
                    payload = pickle.load(f)
                if isinstance(payload, dict) and "data" in payload:
                    vec = ts.ckks_vector_from(context, payload["data"])
                    chunk_groups.setdefault(idx, []).append(vec)
                    print(f"[AGG] Loaded  {client_id}/{fname}")
                else:
                    print(f"[AGG] Invalid format in {fname} — skipped")
            except Exception as e:
                print(f"[AGG] Failed to load {fname}: {e}")

    if not chunk_groups:
        print("[AGG] No valid encrypted chunks to aggregate.")
        return

    # ── Homomorphic aggregation (FedAvg in ciphertext space) ──
    aggregated_chunks = []

    for idx, vectors in sorted(chunk_groups.items()):
        try:
            enc_sum = vectors[0]
            for vec in vectors[1:]:
                if vec.size() != enc_sum.size():
                    print(f"[AGG] Size mismatch at chunk {idx} — skipped")
                    continue
                enc_sum = enc_sum + vec
            enc_avg = enc_sum * (1.0 / len(vectors))
            aggregated_chunks.append((idx, enc_avg))
            print(f"[AGG] Aggregated chunk {idx}  "
                  f"({len(vectors)} clients, avg taken)")
        except Exception as e:
            print(f"[AGG] Failed to aggregate chunk {idx}: {e}")

    # ── Save per-chunk files (optional audit trail) ────────────
    os.makedirs(AGG_CHUNKS_FOLDER, exist_ok=True)
    for idx, enc_chunk in aggregated_chunks:
        path = os.path.join(AGG_CHUNKS_FOLDER, f"agg_chunk_{idx}.bin")
        with open(path, "wb") as f:
            pickle.dump(enc_chunk.serialize(), f)

    print(f"[AGG] {len(aggregated_chunks)} aggregated chunks saved "
          f"→ {AGG_CHUNKS_FOLDER}")

    # ── Write the single global gradient file clients poll for ─
    all_serialized = [chunk.serialize() for _, chunk in aggregated_chunks]
    with open(AGGREGATED_GRAD_FILE, "wb") as f:
        pickle.dump(all_serialized, f)
    print(f"[AGG] Global gradient written → {AGGREGATED_GRAD_FILE}")

    # ── Cleanup received_chunks_bin for next round ─────────────
    try:
        shutil.rmtree(CHUNKS_FOLDER)
        print(f"[AGG] Cleaned up {CHUNKS_FOLDER}")
    except Exception as e:
        print(f"[AGG] Cleanup failed: {e}")


# ── Entry point ───────────────────────────────────────────────
aggregate_encrypted_gradients()

[AGG] CKKS context loaded from /Users/ravi/Desktop/Paper/hybrid-privacy-fl-healthcare/results/context.ser
[AGG] Found 3 client(s): ['hospital_1', 'hospital_2', 'hospital_3']
[AGG] Loaded  hospital_1/chunk_0.bin
[AGG] Loaded  hospital_1/chunk_1.bin
[AGG] Loaded  hospital_2/chunk_0.bin
[AGG] Loaded  hospital_2/chunk_1.bin
[AGG] Loaded  hospital_3/chunk_0.bin
[AGG] Loaded  hospital_3/chunk_1.bin
[AGG] Aggregated chunk 0  (3 clients, avg taken)
[AGG] Aggregated chunk 1  (3 clients, avg taken)
[AGG] 2 aggregated chunks saved → /Users/ravi/Desktop/Paper/hybrid-privacy-fl-healthcare/server/aggregated_chunks_bin
[AGG] Global gradient written → /Users/ravi/Desktop/Paper/hybrid-privacy-fl-healthcare/aggregated_gradient_global_encrypted.pkl
[AGG] Cleaned up /Users/ravi/Desktop/Paper/hybrid-privacy-fl-healthcare/server/received_chunks_bin
